# Cleaning RIPA Datasets
#### Based off of https://github.com/joshuagrossman/ripa/tree/main/src/ripa cleaning scripts

In [3]:
import requests # To download files from the internet
import zipfile # To open zip files
import io # To treat downloaded data like a file in memory
import pandas as pd
import numpy as np

In [4]:
def load_ripa(county):
    """
    Downloads DOJ RIPA Stop Data (2022-2023),
    loads only the Excel file that contains the county name,
    and returns a combined DataFrame.
    """

    # DOJ public ZIP URLs for 2019–2023
    url_dict = {
        2022: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2025-05/RIPA-Stop-Data-2022.zip",
        2023: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2025-05/RIPA-Stop-Data-2023.zip"
    }

    all_years = []

    for year, zip_url in url_dict.items():

        response = requests.get(zip_url)
        response.raise_for_status() # Checks whether website request succeeded

        z = zipfile.ZipFile(io.BytesIO(response.content)) # Opens the zip in memory

        # Find the Excel file containing the county name
        county_file = None # Creates placeholder value
        for file in z.namelist(): # For every file in the ZIP
            if county.lower() in file.lower() and file.lower().endswith(".xlsx"): # Check if file is the right county
                county_file = file
                break

        # If there's no file for that county
        if county_file is None:
            raise ValueError(f"No file found for {county} in {year}")

        # Open the file
        with z.open(county_file) as f:
            df_year = pd.read_excel(f, engine="openpyxl")

        # Make all columns lower-case strings
        df_year.columns = df_year.columns.str.lower()
        df_year["year"] = year # Add year column

        all_years.append(df_year)
        print(f"{year} loaded successfully.")

    final_df = pd.concat(all_years, ignore_index=True)

    return final_df

In [5]:
def load_ripa_2024(county):
    """
    Load DOJ RIPA Stop Data for 2024 for a single county. 
    Kept separate because 2024 is formatted differently, 
    so need to process differently.
    """

    zip_url = "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2025-12/ripa-stop-data-2024.zip"
    
    # Download zip
    response = requests.get(zip_url)
    response.raise_for_status()

    # Open zip in memory
    z = zipfile.ZipFile(io.BytesIO(response.content))

    # Find the Excel file containing the county name
    county_file = None
    for file in z.namelist():
        if file.lower().endswith(".xlsx") and county.lower() in file.lower():
            county_file = file
            break

    if county_file is None:
        raise ValueError(f"No file found for {county} in 2024 zip")

    # Read the county Excel file
    with z.open(county_file) as f:
        df = pd.read_excel(f, engine="openpyxl")

    # Standardize columns + add year
    df.columns = df.columns.str.lower()
    df["year"] = 2024

    print("2024 loaded successfully.")
    return df

In [ ]:
# Read in the 2022-2023 data
orange = load_ripa("Orange")

# Read in 2024 data separately because of updated formatting
orange_24 = load_ripa_2024("Orange")

<positron-console-cell-6>:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


2022 loaded successfully.


<positron-console-cell-6>:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


2023 loaded successfully.


<positron-console-cell-6>:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


2024 loaded successfully.


In [9]:
def add_derived_vars(df):
    """
    Adds the following derived variables:
      - race_ethnicity
      - gender
      - reason_for_contact
      - suspicion
      - action_any_search
      - search_basis_*
      - contraband_weapons
      - contraband_any
      - result_of_stop_arrest

    Returns a new DataFrame (copy) with added columns.
    """

    df = df.copy()

    # ------------------------------------------------------------
    # RACE/ETHNICITY
    # For each row, assign their corresponding race in a new column based on indicator columns
    # Ordered specifically so Hispanic/Latino comes before White, so if both races, labeled Hispanic/Latino
    # 1.2% of the dataset has more than one race listed, which hierarchical ordering handles
    # ------------------------------------------------------------
    
    df["race_ethnicity"] = np.select(
        [
            df["rae_hispanic_latino"] == 1,
            df["rae_black_african_american"] == 1,
            df["rae_asian"] == 1,
            df["rae_middle_eastern_south_asian"] == 1,
            df["rae_pacific_islander"] == 1,
            df["rae_native_american"] == 1,
            df["rae_white"] == 1,
        ],
        [
            "Hispanic",
            "Black",
            "Asian",
            "Middle Eastern/South Asian",
            "Pacific Islander",
            "Native American",
            "White",
        ],
        default="Other/Unknown"
    )

    # ------------------------------------------------------------
    # GENDER
    # ------------------------------------------------------------

    df["gender"] = np.select(
        [
            df["g_gender_nonconforming"] == 1,
            df["g_transgender_woman"] == 1,
            df["g_transgender_man"] == 1,
            df["g_female"] == 1,
            df["g_male"] == 1,
        ],
        [
            "Nonconforming",
            "Transgender Woman",
            "Transgender Man",
            "Female",
            "Male",
        ],
        default="Other/Unknown"
    )

    # ------------------------------------------------------------
    # REASON FOR CONTACT
    # ------------------------------------------------------------

    df["reason_for_contact"] = np.select(
        [
            (df["reason_for_stop"] == 1) & (df["rfs_traffic_violation_type"] == 1),
            (df["reason_for_stop"] == 1) & (df["rfs_traffic_violation_type"] == 2),
            (df["reason_for_stop"] == 1) & (df["rfs_traffic_violation_type"] == 3),
            df["reason_for_stop"] == 2,
            df["reason_for_stop"] == 3,
            df["reason_for_stop"] == 4,
            df["reason_for_stop"] == 6,
            df["reason_for_stop"].isin([5, 7, 8]),
        ],
        [
            "Moving violation",
            "Equipment violation",
            "Non-moving violation",
            "Suspect criminal activity",
            "Parole/probation stop",
            "Outstanding arrest",
            "Consensual search",
            "Truancy/School Related",
        ],
        default="Other/Unknown"
    )

    # ------------------------------------------------------------
    # SUSPICION CATEGORY
    # ------------------------------------------------------------

    df["suspicion"] = np.select(
        [
            df["rfs_rs_off_witness"] == 1,
            df["rfs_rs_match_suspect"] == 1,
            df["rfs_rs_witness_id"] == 1,
            df["rfs_rs_carry_sus_object"] == 1,
            df["rfs_rs_actions_indicative"] == 1,
            df["rfs_rs_suspect_look"] == 1,
            df["rfs_rs_drug_trans"] == 1,
            df["rfs_rs_violent_crime"] == 1,
            df["rfs_rs_reason_susp"] == 1,
        ],
        [
            "Officer witnessed commission of a crime",
            "Matched suspect description",
            "Witness or victim ID of suspect at the scene",
            "Carrying suspicious object",
            "Actions indicative of casing a victim or location",
            "Suspected of acting as a lookout",
            "Actions indicative of a drug transaction",
            "Actions indicative of engaging in a violent crime",
            "Other reasonable suspicion of a crime",
        ],
        default="None"
    )

    # ------------------------------------------------------------
    # ANY SEARCH OCCURRED
    # True if either person search or property search occurred
    # ------------------------------------------------------------

    df["action_any_search"] = (
        (df["ads_search_person"] == 1) |
        (df["ads_search_property"] == 1)
    )

    # ------------------------------------------------------------
    # SEARCH BASES
    # ------------------------------------------------------------

    df["search_basis_plain_view"] = df["bfs_visible_contraband"] == 1
    df["search_basis_plain_smell"] = df["bfs_odor_contraband"] == 1
    df["search_basis_consent"] = df["bfs_consent_given"] == 1
    df["search_basis_safety"] = df["bfs_officer_safety"] == 1
    df["search_basis_suspect_weapon"] = df["bfs_suspect_weapon"] == 1
    df["search_basis_evidence_of_crime"] = df["bfs_evidence"] == 1
    df["search_basis_school_policy"] = df["bfs_school_policy"] == 1
    df["search_basis_emergency"] = df["bfs_exigent_circum"] == 1
    df["search_basis_canine"] = df["bfs_canine_detect"] == 1
    df["search_basis_warrant"] = df["bfs_search_warrant"] == 1
    df["search_basis_probation"] = df["bfs_parole"] == 1
    df["search_basis_incident_to_arrest"] = df["bfs_incident"] == 1
    df["search_basis_vehicle_inventory"] = df["bfs_vehicle_invent"] == 1

    # ------------------------------------------------------------
    # ANY CONTRABAND FOUND
    # ------------------------------------------------------------

    df["contraband_any"] = df["ced_none_contraband"].fillna(0).ne(1)

    # ------------------------------------------------------------
    # ARREST RESULT
    # ------------------------------------------------------------

    df["result_of_stop_arrest"] = (
        (df["ros_custodial_without_warrant"] == 1) |
        (df["ros_custodial_warrant"] == 1)
    )

    # Return a clean copy (prevents pandas fragmentation warnings)
    return df.copy()

In [10]:
def add_search_basis_reasons(df):
    """
    Add two boolean flags indicating whether a stop includes:
      - any discretionary search basis
      - any non-discretionary search basis

    Note: a stop can be True for both if multiple bases are recorded.
    """

    out = df.copy()

    discretionary_cols = [
        "search_basis_plain_view",
        "search_basis_plain_smell",
        "search_basis_consent",
        "search_basis_safety",
        "search_basis_suspect_weapon",
        "search_basis_evidence_of_crime",
        "search_basis_emergency",
        "search_basis_canine",
    ]

    nondiscretionary_cols = [
        "search_basis_warrant",
        "search_basis_probation",
        "search_basis_incident_to_arrest",
        "search_basis_vehicle_inventory",
    ]

    disc_existing = [c for c in discretionary_cols if c in out.columns]
    nondisc_existing = [c for c in nondiscretionary_cols if c in out.columns]

    out["discretionary_search_basis"] = (
        out[disc_existing].fillna(False).any(axis=1) if disc_existing else False
    )

    out["nondiscretionary_search_basis"] = (
        out[nondisc_existing].fillna(False).any(axis=1) if nondisc_existing else False
    )

    out["search_type"] = np.select(
    [
        (out["discretionary_search_basis"]) & (~out["nondiscretionary_search_basis"]),
        (~out["discretionary_search_basis"]) & (out["nondiscretionary_search_basis"]),
        (out["discretionary_search_basis"]) & (out["nondiscretionary_search_basis"]),
    ],
    [
        "Discretionary only",
        "Nondiscretionary only",
        "Mixed",
    ],
    default="No search basis"
    )

    return out

In [11]:
def add_multi_person_stop(df):
    """
    If multiple people were involved in one stop, they all have the same doj_record_id
    but with different person_number
    Add boolean multi-person stop indicator column
    """
    out = df.copy()

    stop_sizes = out.groupby("doj_record_id").size()
    out["multi_person_stop"] = out["doj_record_id"].map(stop_sizes) > 1

    return out

In [12]:
def add_offense_code_digits(df):
    """
    Extract numeric offense codes from text-based offense fields.

    Creates:
      - traffic_violation_cjis_offense_code
      - suspicion_cjis_offense_code

    Example:
        "VC 23152(a) - DUI"  →  "23152"
        "PC 245(a)(1)"       →  "245"
    """

    out = df.copy()

    # Convert to string and extract first sequence of digits
    out["traffic_violation_cjis_offense_code"] = (
        out["rfs_traffic_violation_code"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
    )

    # Reasonable suspicion offense code
    # Do the same thing
    out["suspicion_cjis_offense_code"] = (
        out["rfs_rs_code"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
    )

    return out

In [13]:
# Apply cleaning functions to 2022-2023 data
cleaned_22_23 = add_derived_vars(orange)
cleaned_22_23 = add_search_basis_reasons(cleaned_22_23)
cleaned_22_23 = add_multi_person_stop(cleaned_22_23)
cleaned_22_23 = add_offense_code_digits(cleaned_22_23)

In [14]:
def harmonize_2024_schema(df_24):
    """
    Transform 2024 schema to match 2022-2023 schema structure.
    This allows the same cleaning functions to work on both datasets.
    """
    df = df_24.copy()
    
    # 1. Rename location column
    if 'loc_closest_city' in df.columns:
        df = df.rename(columns={'loc_closest_city': 'closest_city'})
    
    # 2. Map gender columns (2024 → 2022-2023 style)
    # Create the old gender indicator columns from new ones
    if 'g_cisgender_man' in df.columns:
        df['g_male'] = df['g_cisgender_man']
    if 'g_cisgender_woman' in df.columns:
        df['g_female'] = df['g_cisgender_woman']
    if 'g_nonbinary_person' in df.columns:
        df['g_gender_nonconforming'] = df['g_nonbinary_person']
    
    # 3. Map race/ethnicity column
    if 'rae_hispanic_latinex' in df.columns:
        df['rae_hispanic_latino'] = df['rae_hispanic_latinex']
    
    # 4. Combine action columns (nfa_* and ofa_* → ads_*)
    action_mappings = {
        'ads_asked_search_per': 'nfa_asked_search_per',
        'ads_asked_search_prop': 'nfa_asked_search_prop',
        'ads_canine_search': 'nfa_canine_search',
        'ads_curb_detent': 'nfa_curb_detent',
        'ads_patcar_detent': 'nfa_patcar_detent',
        'ads_photo': 'nfa_photo',
        'ads_prop_seize': 'nfa_prop_seize',
        'ads_removed_vehicle_order': 'nfa_removed_vehicle_order',
        'ads_search_pers_consen': 'nfa_search_pers_consent',
        'ads_search_person': 'nfa_search_person',
        'ads_search_prop_consen': 'nfa_search_prop_consent',
        'ads_search_property': 'nfa_search_property',
        'ads_sobriety_test': 'nfa_sobriety_test',
        'ads_vehicle_impound': 'nfa_vehicle_impound',
        'ads_written_statement': 'nfa_written_statement',
    }
    
    for old_col, new_col in action_mappings.items():
        if new_col in df.columns:
            df[old_col] = df[new_col]
    
    # Force actions (ofa_*)
    force_mappings = {
        'ads_baton': ['ofa_baton_drawn', 'ofa_baton_used'],
        'ads_canine_bite': 'ofa_canine_bite',
        'ads_chem_spray': 'ofa_chem_spray',
        'ads_elect_device': ['ofa_elect_device_dart', 'ofa_elect_device_point', 'ofa_elect_device_stun'],
        'ads_firearm_discharge': 'ofa_firearm_discharge',
        'ads_firearm_point': ['ofa_firearm_point', 'ofa_firearm_unholstered'],
        'ads_handcuffed': 'ofa_handcuffed',
        'ads_impact_discharge': ['ofa_impact_projectile_discharge', 'ofa_impact_projectile_point'],
        'ads_removed_vehicle_phycontact': 'ofa_removed_vehicle_phycontact',
    }
    
    for old_col, new_cols in force_mappings.items():
        if isinstance(new_cols, list):
            # Combine multiple columns with OR logic
            matching_cols = [c for c in new_cols if c in df.columns]
            if matching_cols:
                df[old_col] = df[matching_cols].fillna(0).max(axis=1)
        else:
            if new_cols in df.columns:
                df[old_col] = df[new_cols]
    
    # Handle special cases
    if 'nfa_none' in df.columns and 'ofa_none' in df.columns:
        df['ads_no_actions'] = ((df['nfa_none'] == 1) & (df['ofa_none'] == 1)).astype(int)
    
    # 5. Combine warning columns
    if 'ros_verbal_warning' in df.columns and 'ros_written_warning' in df.columns:
        df['ros_warning'] = ((df['ros_verbal_warning'] == 1) | (df['ros_written_warning'] == 1)).astype(int)
    
    # For CDS codes, combine verbal and written warning codes
    if 'ros_verbal_warning_cds' in df.columns and 'ros_written_warning_cds' in df.columns:
        # Take whichever is not null, prioritizing written warnings
        df['ros_warning_cds'] = df['ros_written_warning_cds'].fillna(df['ros_verbal_warning_cds'])
    
    return df

In [15]:
# Harmonize 2024 data to match 2022-2023 schema
orange_24_harmonized = harmonize_2024_schema(orange_24)

# Apply all cleaning functions to orange_24
cleaned_24 = add_derived_vars(orange_24_harmonized)
cleaned_24 = add_search_basis_reasons(cleaned_24)
cleaned_24 = add_multi_person_stop(cleaned_24)
cleaned_24 = add_offense_code_digits(cleaned_24)

In [17]:
# Concatenate using only common columns
common_cols = sorted(set(cleaned_22_23.columns) & set(cleaned_24.columns))

orange_combined = pd.concat([
    cleaned_22_23[common_cols],
    cleaned_24[common_cols]
], ignore_index=True)

In [ ]:
# Save the combined cleaned dataset
orange_combined.to_csv("../data/processed/cleaned_orange_ripa_2022_2024.csv", index=False)
print("Combined dataset saved to: ../data/processed/cleaned_orange_ripa_2022_2024.csv")

Combined dataset saved to: ../data/cleaned/cleaned_orange_ripa_2022_2024.csv
